# E3SM S2S Lead-Time RMSE & Bias Skill Maps (Weeks 1 to 8)

This notebook evaluates **subseasonal error growth, Root Mean Square Error (RMSE), and systematic bias** across **Weeks 1 to 8** (Days 1–56).

### Focus Areas
1. **Spatial Error Growth**: Tracking how forecast error patterns emerge from Day 1–7 (weather memory) to Day 50–56 (climatological saturation).
2. **Systematic Bias**: Lead-dependent mean error $E(L) = \bar{X}(L) - \bar{O}(L)$ identifying rapid shock vs slow drift.
3. **Regional Error Trajectories**: Quantifying RMSE saturation curves across the Tropics, Northern Hemisphere Extratropics, and Southern Hemisphere Extratropics.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

repo_root = Path.cwd()
while repo_root.parent != repo_root and not (repo_root / "esp_lab").is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from esp_lab.paths import figure_output_dir
from esp_lab.diagnostics.s2s_core import (
    S2S_WEEKLY_WINDOWS,
    WEEK_NAMES,
    WEEK_LABELS,
    get_weekly_window,
    compute_weekly_anomalies,
    compute_weekly_rmse,
)
from esp_lab.diagnostics.s2s_io import (
    DEFAULT_DATA_DIR,
    load_s2s_campaign_weekly,
)

print("S2S RMSE & Bias Diagnostic Module Loaded.")

## Configuration and Control Panel

In [ ]:
# =============================================================================
# USER CONTROL PANEL — S2S WEEKLY RMSE & BIAS (WEEKS 1 TO 8)
# =============================================================================

FIELD = "TREFHT"
COMPONENT = "atm"
GRID = "180x360_aave"

LEAD_WEEKS = list(range(1, 9))
INIT_YEARS = list(range(1980, 1987))
INIT_MONTHS = [5, 11]
MEMBERS = [f"EN{i:02d}" for i in range(10)]

E3SM_CASES = {
    "E3SM-4DEnVarOcn": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
        "label": "4DEnVar Ocean Init",
    },
    "E3SM-JRA55_FOSIRL": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "label": "JRA55-FOSIRL Ocean Init",
    },
    "E3SM-Reanalysis": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "label": "Reanalysis (BruteForce)",
    },
}

FIGURE_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")
FIGURE_OUTDIR = figure_output_dir("s2s_skill", COMPONENT, "weekly_rmse", root=FIGURE_ROOT)
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)

print(f"Target Field: {FIELD} ({COMPONENT})")
print(f"Weekly Leads: {LEAD_WEEKS}")
print(f"Figure Output Directory: {FIGURE_OUTDIR}")

## Step 1 — Load S2S Hindcasts (Weeks 1 to 8)

In [ ]:
%%time
model_weekly = {}

for case_key, info in E3SM_CASES.items():
    model_weekly[case_key] = {}
    prefix = info["case_prefix"]
    for m in INIT_MONTHS:
        try:
            da = load_s2s_campaign_weekly(
                data_root=DEFAULT_DATA_DIR,
                case_prefix=prefix,
                years=INIT_YEARS,
                init_month=m,
                members=MEMBERS,
                field=FIELD,
                component=COMPONENT,
                grid=GRID,
                weeks=LEAD_WEEKS,
                verbose=False,
            )
            model_weekly[case_key][m] = da
        except Exception as exc:
            pass

loaded_cases = [k for k, v in model_weekly.items() if len(v) > 0]
print(f"Loaded hindcasts for {len(loaded_cases)} cases: {loaded_cases}")

## Step 2 — Compute Weekly RMSE and Systematic Bias

In [ ]:
%%time
# Use Reanalysis ensemble mean as reference
ref_case = "E3SM-Reanalysis" if "E3SM-Reanalysis" in model_weekly else loaded_cases[0]
rmse_by_case_month = {}
bias_by_case_month = {}

for case_key in loaded_cases:
    rmse_by_case_month[case_key] = {}
    bias_by_case_month[case_key] = {}
    for m in INIT_MONTHS:
        if m in model_weekly[case_key] and m in model_weekly[ref_case]:
            fcst = model_weekly[case_key][m].mean("M", skipna=True)
            obs = model_weekly[ref_case][m].mean("M", skipna=True)
            # RMSE across years
            rmse = np.sqrt(((fcst - obs) ** 2).mean("Y", skipna=True))
            rmse.name = "rmse"
            # Bias across years
            bias = (fcst - obs).mean("Y", skipna=True)
            bias.name = "bias"
            rmse_by_case_month[case_key][m] = rmse
            bias_by_case_month[case_key][m] = bias
            print(f"RMSE computed for {case_key}, Init Month {m:02d}")

print("RMSE and Bias calculations complete.")

## Step 3 — Spatial RMSE Maps Across Weeks 1 to 8

Plots the 2×4 spatial RMSE distribution across Weeks 1 to 8.

In [ ]:
%%time
def plot_weekly_rmse_maps(rmse_da, case_name, init_month, field_name):
    fig, axes = plt.subplots(
        nrows=2, ncols=4, figsize=(20, 9),
        subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)}
    )
    axes = axes.flatten()
    max_val = float(rmse_da.quantile(0.98))
    levels = np.linspace(0, max_val, 21)
    month_name = {5: "May", 11: "November"}.get(init_month, f"Month {init_month}")

    for idx, w in enumerate(range(1, 9)):
        ax = axes[idx]
        ax.coastlines(linewidth=0.8, color="0.2")
        ax.set_global()
        if w in rmse_da.L.values:
            rw = rmse_da.sel(L=w)
            cf = ax.contourf(
                rw.lon, rw.lat, rw,
                levels=levels, cmap="magma_r", extend="max",
                transform=ccrs.PlateCarree()
            )
        w_def = get_weekly_window(w)
        ax.set_title(w_def.label, fontsize=12, fontweight="bold")

    cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.025])
    cbar = fig.colorbar(cf, cax=cbar_ax, orientation="horizontal")
    cbar.set_label(f"{field_name} RMSE", fontsize=12)

    fig.suptitle(
        f"{case_name} — {field_name} Subseasonal Weekly RMSE (Weeks 1–8)\nInitialized {month_name}",
        fontsize=16, fontweight="bold", y=0.98
    )
    plt.subplots_adjust(bottom=0.12, top=0.92, hspace=0.15, wspace=0.08)

    out_path = FIGURE_OUTDIR / f"{case_name}_{field_name}_init{init_month:02d}_rmse_w1_w8.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"Figure saved: {out_path}")
    plt.show()

for case_key in rmse_by_case_month:
    for m in rmse_by_case_month[case_key]:
        plot_weekly_rmse_maps(
            rmse_by_case_month[case_key][m],
            case_name=case_key,
            init_month=m,
            field_name=FIELD,
        )

## Step 4 — Regional Error Growth Curves (Weeks 1 to 8)

Plots the trajectory of RMSE growth from Week 1 to Week 8 for Tropics, Northern Hemisphere Extratropics, and Southern Hemisphere Extratropics.

In [ ]:
%%time
regions = {
    "Global": (-90, 90),
    "Tropics": (-20, 20),
    "NH Extratropics": (20, 80),
    "SH Extratropics": (-80, -20),
}

plt.figure(figsize=(10, 6))
for case_key in rmse_by_case_month:
    for m in rmse_by_case_month[case_key]:
        da = rmse_by_case_month[case_key][m]
        for rname, (s_lat, n_lat) in regions.items():
            reg_mean = da.sel(lat=slice(s_lat, n_lat)).mean(["lat", "lon"])
            plt.plot(reg_mean.L, reg_mean, marker="o", label=f"{case_key} {rname} (m={m:02d})")

plt.xlabel("Forecast Lead Week", fontsize=12)
plt.ylabel(f"{FIELD} RMSE", fontsize=12)
plt.title(f"{FIELD} Subseasonal Error Growth Curves (Weeks 1 to 8)", fontsize=14, fontweight="bold")
plt.xticks(range(1, 9), [f"W{w}" for w in range(1, 9)])
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

growth_path = FIGURE_OUTDIR / f"{FIELD}_rmse_growth_curves_w1_w8.png"
plt.savefig(growth_path, dpi=200, bbox_inches="tight")
print(f"Growth curves saved: {growth_path}")
plt.show()

## Validation & Integrity Check

In [ ]:
for case_key in rmse_by_case_month:
    for m in rmse_by_case_month[case_key]:
        da = rmse_by_case_month[case_key][m]
        assert "L" in da.dims, f"L dimension missing in {case_key}"
        assert len(da.L) == 8, f"Expected 8 weekly leads in {case_key}"
        assert (da.values[~np.isnan(da.values)] >= 0).all(), "Negative RMSE values found!"

print("Validation SUCCESS: All 8 weekly leads verified with non-negative RMSE.")